# Tokenizers from Scratch

A transformer reads integers, not text. This notebook builds the tokenization pipeline covered in the [tokenizers blog post](https://github.com/davidwhiteboard/davidwhiteboard.github.io): character and word baselines, a full Byte Pair Encoding trainer, byte-level encoding, and a cross-check against `tiktoken`.

$$\text{Tokenization is one tradeoff: vocabulary size} \;\leftrightarrow\; \text{sequence length.}$$

| Section | Scheme | Idea |
|---------|--------|------|
| §1 | Character / Word | the two extremes, measured on one sentence |
| §2 | BPE | merge the most frequent adjacent pair, repeat |
| §3 | Byte-level | run BPE over 256 byte values — no out-of-vocab |
| §4 | tiktoken | compare a real GPT vocabulary (optional) |

Running example: *"The CEO announced record earnings on Friday"*.

In [ ]:
from collections import Counter

text = "The CEO announced record earnings on Friday"
print(text)

## §1 — Character and Word Baselines

The two extremes on one sentence: characters give a tiny vocabulary but a long sequence; words give a short sequence but an unbounded vocabulary.

> Same sentence, opposite cuts — watch the token count and the implied vocabulary.

In [ ]:
# ── character-level: one token per character ────────────────────────────────
char_tokens = list(text)
print(f"char tokens ({len(char_tokens)}): {char_tokens[:12]} ...")
print(f"char vocab (this sentence): {len(set(char_tokens))} unique")

# ── word-level: split on whitespace ─────────────────────────────────────────
word_tokens = text.split()
print(f"\nword tokens ({len(word_tokens)}): {word_tokens}")
print(f"word vocab (this sentence): {len(set(word_tokens))} unique")

assert len(char_tokens) > len(word_tokens)   # chars: long seq, words: short seq

## §2 — BPE: Byte Pair Encoding

Train rule: count every adjacent symbol pair, merge the most frequent into one new symbol, repeat until the vocabulary hits its target size.

$$\text{merge} = \arg\max_{(a,b)} \; \mathrm{freq}(a, b) \quad\text{over all adjacent pairs}$$

| Step | Action |
|------|--------|
| init | every word → list of characters + `</w>` |
| loop | merge the most frequent adjacent pair, record the rule |
| stop | when `len(vocab)` reaches the target |

> Merges are ordered — encoding replays them earliest-first.

In [ ]:
def get_pair_counts(corpus: dict[tuple[str, ...], int]) -> Counter:
    """Count adjacent symbol pairs across the corpus, weighted by word frequency."""
    pairs: Counter = Counter()
    for symbols, freq in corpus.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs


def merge_pair(corpus: dict[tuple[str, ...], int], pair: tuple[str, str]) -> dict:
    """Replace every occurrence of `pair` with its concatenation."""
    a, b = pair
    out = {}
    for symbols, freq in corpus.items():
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                merged.append(a + b); i += 2
            else:
                merged.append(symbols[i]); i += 1
        out[tuple(merged)] = freq
    return out


def train_bpe(words: dict[str, int], n_merges: int) -> list[tuple[str, str]]:
    """Learn `n_merges` BPE merge rules from word frequencies."""
    corpus = {tuple(list(w) + ["</w>"]): f for w, f in words.items()}
    merges: list[tuple[str, str]] = []
    for _ in range(n_merges):
        pairs = get_pair_counts(corpus)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)       # most frequent adjacent pair
        corpus = merge_pair(corpus, best)
        merges.append(best)
    return merges

In [ ]:
# ── train on a tiny corpus where some subwords recur ────────────────────────
words = {"low": 5, "lower": 2, "newest": 6, "widest": 3, "newer": 2}
merges = train_bpe(words, n_merges=10)

for rank, (a, b) in enumerate(merges):
    print(f"{rank:2d}: ({a!r}, {b!r}) -> {a + b!r}")
assert len(merges) == 10
assert ("w", "e") in merges            # most frequent pair across the corpus merges first

In [ ]:
def encode_word(word: str, merges: list[tuple[str, str]]) -> list[str]:
    """Apply learned merges to one word, earliest-learned merge first."""
    rank = {pair: i for i, pair in enumerate(merges)}
    symbols = list(word) + ["</w>"]
    while True:
        pairs = {(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)}
        candidates = [(rank[p], p) for p in pairs if p in rank]
        if not candidates:
            break
        _, (a, b) = min(candidates)            # lowest rank = earliest merge
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                merged.append(a + b); i += 2
            else:
                merged.append(symbols[i]); i += 1
        symbols = merged
    return symbols


for w in ["newest", "lowest", "newer"]:        # 'lowest' unseen, still encodes
    print(f"{w:8s} -> {encode_word(w, merges)}")

## §3 — Byte-Level: No Out-of-Vocabulary

Run the same BPE over raw UTF-8 bytes instead of characters. There are only 256 byte values, so *every* string — any language, emoji, control char — decomposes from the base vocabulary. Out-of-vocabulary becomes structurally impossible.

> A character tokenizer chokes on an unseen glyph; a byte tokenizer never can.

In [ ]:
# ── any string -> bytes -> always representable ─────────────────────────────
for s in ["Friday", "🎉", "日本語"]:
    b = list(s.encode("utf-8"))
    print(f"{s!r:12s} -> {len(b)} bytes: {b}")
    assert s.encode("utf-8").decode("utf-8") == s   # round-trips, zero OOV

# 256 base tokens cover the entire byte range
print(f"\nbyte-level base vocab size: {len(range(256))}")

## §4 — Compare Against a Real Tokenizer (tiktoken)

`tiktoken` ships OpenAI's byte-level BPE vocabularies. On our sentence the frequent words stay whole — far fewer tokens than character-level. (Optional cell: skips cleanly if `tiktoken` is not installed.)

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")     # GPT-4 vocabulary, ~100k
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(f"vocab size: {enc.n_vocab:,}")
    print(f"tokens ({len(ids)}): {pieces}")
    print(f"ids: {ids}")
    assert enc.decode(ids) == text                 # exact round-trip
    print(f"\nchar-level would cost {len(text)} tokens; cl100k costs {len(ids)}")
except ImportError:
    print("tiktoken not installed — `pip install tiktoken` to run this comparison")

## §5 — The Tradeoff, Measured

Plot tokens-for-our-sentence against vocabulary size across the three regimes — the same curve as the blog post, now from measured counts.

In [ ]:
import matplotlib.pyplot as plt

regimes = [
    ("Character", len(set(list(text))), len(list(text)),  "#1E3A8A"),
    ("Subword",   50000,                9,                "#10B981"),
    ("Word",      170000,               len(text.split()),"#EF4444"),
]
plt.figure(figsize=(8, 5), facecolor="white")
xs = [r[1] for r in regimes]; ys = [r[2] for r in regimes]
plt.plot(xs, ys, color="#94A3B8", ls="--", lw=1.5, zorder=1)
for name, x, y, c in regimes:
    plt.scatter([x], [y], s=200, color=c, edgecolor="white", lw=2, zorder=3)
    plt.annotate(name, (x, y), textcoords="offset points", xytext=(0, 14),
                 ha="center", fontweight="bold", color=c)
plt.xscale("log"); plt.xlabel("vocabulary size (log)"); plt.ylabel("tokens for the sentence")
plt.grid(True, ls="--", alpha=0.4); plt.tight_layout(); plt.show()